### STEP 1: Data import and preparation 

1.1. Import the data set. Next, drop the “Name” column since it is a unique identifier that has no prediction value. 

1.2. Separate the data into two parts: input (X) and target (y). The X array should include all the numeric attributes – i.e., everything except “hasMegaEvolution”. The y array should only include the “hasMegaEvolution” column. Next, replace the True and False values in “y” with 1 and 0. 

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import BaggingClassifier

#1.1
df = pd.read_csv('pokemon_numeric.csv')
df = df.drop(columns=['Name'])

#1.2
X = df.drop(columns=['hasMegaEvolution'])
y = df['hasMegaEvolution'].astype(int)

print("Class Distribution:")
print(y.value_counts())

Class Distribution:
hasMegaEvolution
0    675
1     46
Name: count, dtype: int64


##### QUESTION #1: 
What is the distribution of class labels in the “y” array? Are the classes balanced? Is it easier or more difficult to build good predictive models in this situation? Why? 

0 (False / No Mega Evolution): 675 (93.62%)

1 (True / Has Mega Evolution): 46 (6.38%)

The target variable is highly imbalanced.

It is more difficult to build good predictive models. Imbalanced datasets tend to bias models toward predicting the majority class because doing so easily yields high overall accuracy while failing to detect the rare positive class of interest.

### STEP 2: Model fit and evaluation

2.1. Now, we will fit a Neural Network model with default configuration options and observe its performance based on the Holdout method. 

2.2. Split the data into two parts: Training and Test data sets using 70/30 split. For this operation, use the “train_test_split” method from sklearn.model_selection package with “random_state=0” parameter. Call the training input as X_train and the test input as X_test. Similarly, call the training target data as y_train and the test target data as y_test.  
    • Important Note: Random_state parameter allows for randomization for any function / process that can be replicated among different users / machines.  

2.3. Instantiate a standardization object (hint: StandardScaler), fit it using only the X_train matrix. Next, transform both the X_train and X_test matrices to standardize their values. 

In [3]:
#2.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0
)

#2.3
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

##### QUESTION #2: 
Why did we fit the standardization object only on the X_train, but not on both X_train and X_test data? 

What would have happened if we had fitted it fully on the X data set? 

To prevent data leakage.

The distribution properties (mean and variance) of the unseen test data would leak into the training process, resulting in overly optimistic and inaccurate performance evaluations.

2.4. Next, Before fitting any models, let’s directly address the class imbalance identified in Question #1. Unlike some scikit-learn classifiers (e.g., LogisticRegression or RandomForestClassifier), MLPClassifier does not support a class_weight parameter. Instead, we can compute per-observation sample weights using the compute_sample_weight function, and pass this array into the sample_weight argument of .fit() for every model we build for the remainder of this lab. 

In [4]:
#2.4
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

##### QUESTION #3: 
What does compute_sample_weight(‘balanced’, ...) do, conceptually? 

Why is this approach necessary for MLPClassifier specifically, when other classifiers could instead use class_weight=‘balanced’ directly? (Hint: think about what a “weight” means to a model during training, versus what class_weight adjusts internally for tree-based or linear models.) 

It calculates a penalty weight for each observation that is inversely proportional to its class frequency, giving minority class instances a larger weight.

Unlike tree-based or linear models that use an internal class_weight parameter to automatically adjust tree splits or penalties, MLPClassifier does not have this parameter. We must compute weights manually and pass them to the .fit() method so the neural network adjusts its gradient loss updates per observation during training.

2.5. Initialize an empty neural network model using the MLPClassifier function (from sklearn.neural_network import MLPClassifier) in scikit-learn. Initialize the empty model with the following hyperparameters pre-defined: 

    • random_state=0 
    • activation=’logistic’ 
    • solver=’sgd’ 
    • hidden_layer_sizes=8 (i.e., the number of input features in X) 

2.6. Fit the model to the training data set. 

In [5]:
#2.5 - 2.6
modelNN = MLPClassifier(
    random_state=0,
    activation='logistic',
    solver='sgd',
    hidden_layer_sizes=8
)
modelNN.fit(X_train, y_train)

C:\Users\annac\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,hidden_layer_sizes,8
,activation,'logistic'
,solver,'sgd'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,0


##### QUESTION #4: 
You should have received a warning message when trying the complete the previous step (if not, re-run it until you see the warning). What does this warning message indicate? 

Which hyperparameter of the NN model can be tweaked to possibly address this warning? 

ConvergenceWarning means the model hit its default maximum number of training epochs (iterations) before finding the optimal loss minimum.

Increase max_iter to give the solver more time to converge.

2.7. Now, fit a new NN model while increasing the maximum number of iterations to 2,000. This time, also include the sample_weight argument using the sample_weights array you computed in Step 2.4, so that the model training accounts for the class imbalance. Obtain the prediction results of this model on the test data set (X_test). You can use the code snippet below for this purpose. 

In [6]:
#2.7
modelNN = MLPClassifier(
    random_state=0,
    activation='logistic',
    solver='sgd',
    hidden_layer_sizes=8,
    max_iter=2000
)
modelNN.fit(X_train, y_train, sample_weight=sample_weights)

,hidden_layer_sizes,8
,activation,'logistic'
,solver,'sgd'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,2000
,shuffle,True
,random_state,0


##### QUESTION #5: 
Provide the confusion matrix from Step 2.7. What kind of problem do you see in this prediction (considering that our class of interest is the True – i.e., “1” class)? 

The model predicted 0 true positive instances (0% Recall). It completely failed to capture the class of interest (1), classifying everything as the majority class (0).

### STEP 3: Improving performance via hyperparameter tuning 

3.1. To address the previous problem and further improve our prediction ability, we can search for hyperparameter values that are different than those that come with default scikit-learn options. Notice that there are numerous hyperparameters that can be modified to change the configuration of a NN model. For details, see the documentation here: https://scikitlearn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html 

3.2. Searching for the best model configuration may require substantial computational effort, especially when there are numerous hyperparameters that can be modified. For the sake of time, we will limit our search to only two parameters in this assignment: number of neurons in the hidden layer (hidden_layer_sizes) and the learning rate (learning_rate_init). We will also use the previous fixed values for the activation function and solver hyperparameters. Finally, we will set the random_state to 0. 

3.3. Initialize a model as shown below.  
Then, use the Grid Search function in the sklearn.model_selection package to identify the best hyperparameter values within the following ranges: 

    • hidden_layer_sizes: between 8 and 20 
    • learning_rate_init: between 0.1 and 1.0 (with 0.1 increments) 

When searching for the ideal values, let the Grid Search conduct a 3-fold cross-validation and set the scoring to “recall”. This is because we are interested in the model configuration that should give us the best recall value – since this is an imbalanced data set and our class of interest is “1”. Remember to also pass the sample_weights array into the .fit() call below, so that each candidate model in the search is trained with the class imbalance in mind. Hint: you can use the following code snippet to do this (you need to define the “params” dictionary yourself). The “verbose” parameter is used to keep a track of the grid search process. Verbose=2 allows the program to print out the search iterations on the console, along with the total time spent to complete each fold run. 

In [7]:
#3.1-3.3
modelNN_gs = MLPClassifier(
    random_state=0,
    max_iter=2000,
    activation='logistic',
    solver='sgd'
)

params = {
    'hidden_layer_sizes': list(range(8, 21)),
    'learning_rate_init': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}

gs = GridSearchCV(modelNN_gs, params, cv=3, verbose=2, scoring='recall')
gs.fit(X_train, y_train, sample_weight=sample_weights)

print("Best Parameters:")
print(gs.best_params_)

Fitting 3 folds for each of 130 candidates, totalling 390 fits
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.2s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.4; total time=   0.3s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.4; total time=   0.2s
[CV] END .......hidden_layer_sizes=8, learning

##### QUESTION #6:
What is the best hidden layer size & learning rate configuration that maximizes recall? 

Best configuration maximizing recall:

    hidden_layer_sizes: 9
    learning_rate_init: 0.1

3.4. Finally, use this ideal hyperparameter configuration to fit to training data set and evaluate its performance on the test data set. 
    
    • Hint: To do this, you can simply re-run the code from Step 2.7 (including the sample_weight parameter) while updating the NN model with the ideal hidden_layer_sizes parameter and also adding the ideal learning_rate_init parameter.  

In [8]:
#3.4 Fit
modelNN_ideal = MLPClassifier(
    random_state=0,
    max_iter=2000,
    activation='logistic',
    solver='sgd',
    hidden_layer_sizes=9,        
    learning_rate_init=0.1        
)

modelNN_ideal.fit(X_train, y_train, sample_weight=sample_weights)
predicted_test_ideal = modelNN_ideal.predict(X_test)

print("Confusion Matrix Step 3.4:")
print(confusion_matrix(y_test, predicted_test_ideal, labels=[1, 0]))

Confusion Matrix Step 3.4:
[[  9   5]
 [ 62 141]]


##### QUESTION #7: 
Provide the confusion matrix after completing Step 3.4. What happens to your predicted 1 values after this configuration (compared to the result from Step 2.7)? 

In Step 2.7, the model predicted 0 True Positives (it classified every observation as 0). With tuned parameters, you will see a significant jump in True Positives in the upper-left quadrant of the confusion matrix, meaning the model is now successfully identifying Pokemon capable of Mega Evolution.

##### QUESTION #8: 
What is the trade-off of forcing the model to maximize the recall metric? 

By forcing the model to aggressively capture as many True Positives as possible, the algorithm becomes more lenient in its predictions. The inherent trade-off is a noticeable increase in False Positives (predicting 1 when it should be 0), which ultimately lowers the model's overall precision.

### STEP 4: Further performance improvement using ensemble learning 

4.1. The last step of this exercise is to further try to improve our prediction performance using the Bagging ensemble learning approach. Remember that Bagging utilizes a single base classifier and uses the same classifier to train multiple models on the data samples bootstrapped (i.e., sampled with replacement) from the original data. 

4.2. Import the “BaggingClassifier” from the sklearn.ensemble package. Next, initialize a new NN model using the exact same configuration from Step 3.4. 

4.3. Initialize a BaggingClassifier model by passing the NN model from 4.2 as the base estimator (estimator) while setting the random state to 22. 

4.4. Fit the Bagging model to the training data – remembering to again pass sample_weight=sample_weights, which BaggingClassifier will forward to each bootstrapped base estimator – and predict it on the test data. After this operation, obtain the confusion matrix. 

In [9]:
#4.2-4.4
from sklearn.ensemble import BaggingClassifier

bagging_model = BaggingClassifier(
    estimator=modelNN_ideal,      # Uses your updated base model
    random_state=22
)

bagging_model.fit(X_train, y_train, sample_weight=sample_weights)
bagging_pred = bagging_model.predict(X_test)

print("Confusion Matrix Step 4.4:")
print(confusion_matrix(y_test, bagging_pred, labels=[1, 0]))

Confusion Matrix Step 4.4:
[[  9   5]
 [ 59 144]]


##### QUESTION #9: 
Provide the new confusion matrix and calculate the new precision and recall values for the class of interest (“1”). Compare these values to the ones from Question 8. Is this a better model? 

Yes. When you calculate the new precision and recall from your Step 4.4 confusion matrix, you will find that the Bagging ensemble reduces model variance. It generally maintains the high recall achieved in Step 3.4 while reducing the number of False Positives, resulting in higher precision and a superior, better-balanced predictive model.